# Udemy Course Data Analysis

Explore course pricing, ratings, course length, and student engagement, then evaluate a linear regression baseline for predicting course prices.

**Dataset:** Add `Course_info.csv` to `data/` before running the notebook. See the [README](README.md) for setup instructions and the Kaggle source reference.

**Workflow:** load and validate data -> inspect quality -> explore trends -> train and evaluate a regression baseline.


## 1. Load the data

This analysis uses the course metadata fields listed below. If a column is missing, the notebook stops with an explicit message instead of silently producing partial results.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_PATH = Path("data") / "Course_info.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Follow the README instructions to add Course_info.csv."
    )

courses = pd.read_csv(DATA_PATH, encoding="utf-8")
courses.columns = courses.columns.str.strip()

required_columns = {
    "is_paid",
    "price",
    "num_subscribers",
    "avg_rating",
    "num_reviews",
    "num_lectures",
    "content_length_min",
    "category",
    "subcategory",
    "language",
}
missing_columns = sorted(required_columns.difference(courses.columns))
if missing_columns:
    raise ValueError(f"Course_info.csv is missing required columns: {missing_columns}")

print(f"Rows: {len(courses):,} | Columns: {courses.shape[1]:,}")
display(courses.head())


## 2. Data quality and preparation

Review missing values and duplicates first. Numeric fields are parsed explicitly; rows without a valid, non-negative price are excluded from the price analysis. Missing feature values are imputed inside the modeling pipeline.


In [ ]:
numeric_columns = [
    "price",
    "avg_rating",
    "num_reviews",
    "num_subscribers",
    "num_lectures",
    "content_length_min",
]

quality = pd.DataFrame(
    {
        "missing_values": courses.isna().sum(),
        "missing_percent": (courses.isna().mean() * 100).round(2),
    }
).sort_values("missing_values", ascending=False)
display(quality.head(15))

for column in numeric_columns:
    courses[column] = pd.to_numeric(courses[column], errors="coerce")

courses[numeric_columns] = courses[numeric_columns].replace([np.inf, -np.inf], np.nan)
duplicate_key = "id" if "id" in courses.columns else None
duplicate_count = courses.duplicated(subset=duplicate_key).sum()
analysis = courses.drop_duplicates(subset=duplicate_key).copy()
valid_price = analysis["price"].notna() & analysis["price"].ge(0)
invalid_price_count = int((~valid_price).sum())
analysis = analysis.loc[valid_price].copy()

print(f"Duplicate rows removed: {duplicate_count:,}")
print(f"Rows without a valid non-negative price removed: {invalid_price_count:,}")
print(f"Rows retained for analysis: {len(analysis):,}")


In [ ]:
display(analysis[numeric_columns].describe().T.round(2))


## 3. Exploratory analysis

The charts below summarize price and rating distributions and compare engagement across course categories. They are descriptive: associations in this dataset do not establish cause and effect.


In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=analysis, x="price", bins=40, ax=axes[0])
axes[0].set(title="Course price distribution", xlabel="Price", ylabel="Course count")

sns.histplot(data=analysis, x="avg_rating", bins=30, ax=axes[1])
axes[1].set(title="Course rating distribution", xlabel="Average rating", ylabel="Course count")

plt.tight_layout()
plt.show()


In [ ]:
paid_counts = analysis["is_paid"].astype("string").fillna("Unknown").value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=paid_counts.index, y=paid_counts.values, ax=ax, color="steelblue")
ax.set(title="Courses by paid status", xlabel="Paid status", ylabel="Course count")
plt.tight_layout()
plt.show()


In [ ]:
top_categories = (
    analysis.groupby("category", dropna=False)["num_subscribers"]
    .sum()
    .nlargest(10)
    .sort_values()
)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=top_categories.values, y=top_categories.index.astype(str), ax=ax, color="steelblue")
ax.set(title="Top 10 categories by total subscribers", xlabel="Subscribers", ylabel="Category")
plt.tight_layout()
plt.show()


In [ ]:
median_price_by_category = (
    analysis.groupby("category")["price"]
    .median()
    .nlargest(10)
    .sort_values()
)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    x=median_price_by_category.values,
    y=median_price_by_category.index.astype(str),
    ax=ax,
    color="darkorange",
)
ax.set(title="10 categories with the highest median price", xlabel="Median price", ylabel="Category")
plt.tight_layout()
plt.show()


In [ ]:
correlation_columns = [
    "price",
    "avg_rating",
    "num_reviews",
    "num_subscribers",
    "num_lectures",
    "content_length_min",
]
correlations = analysis[correlation_columns].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation between numeric course attributes")
plt.tight_layout()
plt.show()


## 4. Price prediction with linear regression

Predict course price from course metadata: paid status, category, subcategory, language, number of lectures, and course length. Ratings, reviews, and subscriber counts are excluded from the model. Numeric and categorical preprocessing is fitted on training data only to avoid leaking test-set information.

Compare the model with a median-price baseline. MAE and RMSE are in the dataset's price units; higher R? indicates more variance explained on this test split. A negative R? means the model performed worse than predicting the test-set mean.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["num_lectures", "content_length_min"]
categorical_features = ["is_paid", "category", "subcategory", "language"]
feature_columns = numeric_features + categorical_features
model_data = analysis.dropna(subset=["price"])

if len(model_data) < 10:
    raise ValueError("At least 10 rows with valid prices are required for a train/test split.")

X = model_data[feature_columns]
y = model_data["price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)
baseline = DummyRegressor(strategy="median")
model.fit(X_train, y_train)
baseline.fit(X_train, y_train)
model_predictions = model.predict(X_test)
baseline_predictions = baseline.predict(X_test)

metrics = pd.DataFrame(
    {
        "model": ["Median baseline", "Linear regression"],
        "MAE": [
            mean_absolute_error(y_test, baseline_predictions),
            mean_absolute_error(y_test, model_predictions),
        ],
        "RMSE": [
            np.sqrt(mean_squared_error(y_test, baseline_predictions)),
            np.sqrt(mean_squared_error(y_test, model_predictions)),
        ],
        "R2": [
            r2_score(y_test, baseline_predictions),
            r2_score(y_test, model_predictions),
        ],
    }
).set_index("model")
display(metrics.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(x=y_test, y=model_predictions, alpha=0.6, ax=ax)
plot_min = min(float(y_test.min()), float(model_predictions.min()))
plot_max = max(float(y_test.max()), float(model_predictions.max()))
ax.plot([plot_min, plot_max], [plot_min, plot_max], linestyle="--", color="black")
ax.set(
    title="Actual vs. predicted course prices",
    xlabel="Actual price",
    ylabel="Predicted price",
)
plt.tight_layout()
plt.show()


In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["regressor"].coef_
coefficient_summary = pd.DataFrame(
    {"feature": feature_names, "coefficient": coefficients}
)
coefficient_summary["absolute_coefficient"] = coefficient_summary["coefficient"].abs()
display(
    coefficient_summary.nlargest(10, "absolute_coefficient")
    .drop(columns="absolute_coefficient")
    .reset_index(drop=True)
)


## 5. Interpretation and limitations

Use the metric table and actual-versus-predicted plot to judge whether the linear model improves on the baseline. Coefficients describe model associations for this fitted sample; they are not causal effects. One-hot encoded category coefficients are relative to the encoder's reference behavior.

This dataset is a snapshot, so prices and course popularity may have changed. The random holdout is a simple baseline evaluation, and a linear model may not capture nonlinear relationships. For a stronger study, test alternative models, use cross-validation, and evaluate against a later time period if timestamps are suitable.
